# Setup

- macOS: Run 'brew install poppler'
- Ubuntu/Debian: Run 'apt-get install poppler-utils'
- Windows: Install from https://github.com/oschwartz10612/poppler-windows/releases/ and add to PATH

In [ ]:
# !pip install typhoon-ocr==0.4.1
# !pip install python-dotenv
# !pip install lxml
# !pip install html5lib

In [ ]:
from dotenv import load_dotenv
import os
import pandas as pd
from typhoon_ocr import ocr_document
from io import StringIO
import re

import sys
import os

sys.path.append(os.path.abspath(".."))
from utils import *

load_dotenv()

In [ ]:
KEYWORD = "district"

PARTICIPANTS_DISTRICT = [
    [
        "นายภาคภูมิ โภคทรัพย์", 
        "นายพีรภัทร ทองธีรสกุล", 
        "นายธนายุทธ ยืนยั่ง", 
        "นางสุทธิลักษณ์ ยายิรัมย์", 
        "นายสนอง เทพอักษรณรงค์", 
        "นายวิเชียร ลานทอง", 
        "นายนาท ฉัพพรรณธนกูร", 
        "นายอิทธิพัทธ์ ภักดีเนติพันธุ์"
    ],
    
    [
        "ประชาธิปัตย์", 
        "เพื่อไทย", 
        "ประชาชน", 
        "รวมไทยสร้างชาติ", 
        "ภูมิใจไทย", 
        "ประชากรไทย", 
        "กล้าธรรม", 
        "เศรษฐกิจ"
    ],
]

# OCR

In [ ]:
API_KEY = os.getenv("TYPHOON_OCR_API_KEY")

### Reader Function

In [ ]:
# อ่านข้อมูล file ตระกูลสส.เขต
def read_district(file_name: str, page_num: int):
    print(f"Processing District - File: {file_name} - Page: {page_num}")
    safe_path = get_safe_pdf_path(file_name)

    data_json = ocr_document(
        api_key=API_KEY,
        pdf_or_image_path=safe_path,
        page_num=page_num,
        target_image_dim=2400,
    )
    
    data_json = thai_to_arabic(data_json)
    
    all_card = re.search(
        r'(?:ได้รับบัตรเลือกตั้ง.*?ทั้งหมด|จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร.*?จำนวน)\s*(\d[\d,]*)\s*บัตร',
        data_json
    )
    good_card = re.search(r'บัตรดี.*?(\d[\d,]*)\s*บัตร', data_json)
    bad_card = re.search(r'บัตรเสีย.*?(\d[\d,]*)\s*บัตร', data_json)
    none_card = re.search(r'บัตรที่ไม่เลือกผู้สมัครผู้ใด.*?(\d[\d,]*)\s*บัตร', data_json)

    summary = {
        'จำนวนบัตรทั้งหมด': parse_number(all_card.group(1)) if all_card else 0,
        'บัตรดี': parse_number(good_card.group(1)) if good_card else 0,
        'บัตรเสีย': parse_number(bad_card.group(1)) if bad_card else 0,
        'ไม่เลือกผู้ใด': parse_number(none_card.group(1)) if none_card else 0,
    }
    
    df = pd.read_html(StringIO(data_json))[0]
        
    df.columns = ['หมายเลข', 'ชื่อผู้สมัคร', 'พรรค', 'คะแนน']
    df = df.iloc[1:].reset_index(drop=True)
    df.drop(columns=['หมายเลข'], inplace=True)
    
    # ตัดแถวให้เหลือแค่พรรคที่มี
    df = df.head(len(PARTICIPANTS_DISTRICT[0]))
    
    # ถ้าค่าใน column พรรค เป็นตัวเลข ให้เอามาใส่แทนที่ใน column คะแนน
    df[['พรรค', 'คะแนน']] = df.apply(swap_if_party_is_numeric, axis=1)[['พรรค', 'คะแนน']]
    
    # เติมชื่อผู้สมัครและพรรค
    df["ชื่อผู้สมัคร"] = PARTICIPANTS_DISTRICT[0]
    df["พรรค"] = PARTICIPANTS_DISTRICT[1]

    # Extract เฉพาะตัวเลขจากคะแนน
    df['คะแนน'] = (df['คะแนน']
                    .astype(str)
                    .str.replace(',', '')
                    .str.findall(r'\d+') 
                    .str[-1]
                    .pipe(pd.to_numeric, errors='coerce')
                    .fillna(0)
                    .astype(int))
    
    # เก็บผลลัพธ์
    save_ocr_output(file_name, page_num, summary, df)
    
    return summary, df

### Reader

In [ ]:
for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if KEYWORD in file_config["path"]:
        for page_num in range(1, total_pages, OCR_STEPS[KEYWORD]):
            read_district(file_config["path"], page_num)

# Check OCR Result

In [ ]:
dfs = []

for file_config in OCR_FILES:
    total_pages = file_config["pages"]
    if KEYWORD in file_config["path"]:
        for page_num in range(1, total_pages, OCR_STEPS[KEYWORD]):
            df_dict = load_ocr_data(file_config["path"], page_num)
            dfs.append(df_dict)

In [ ]:
dfs[0]